<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/FramePack_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/FramePack_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎞️ FramePack & FramePack-F1 — Next-Frame-Prediction 60s Constant-VRAM Video Generation

**Models:** [`lllyasviel/FramePack_F1_I2V_HY_20250503`](https://huggingface.co/lllyasviel/FramePack_F1_I2V_HY_20250503) &nbsp;|&nbsp; [`lllyasviel/FramePackI2V_HY`](https://huggingface.co/lllyasviel/FramePackI2V_HY) &nbsp;|&nbsp; **Shared Encoders:** [`hunyuanvideo-community/HunyuanVideo`](https://huggingface.co/hunyuanvideo-community/HunyuanVideo) &nbsp;|&nbsp; **GitHub:** [`lllyasviel/FramePack`](https://github.com/lllyasviel/FramePack) &nbsp;|&nbsp; **License:** Apache 2.0

**FramePack** (by Lvmin Zhang & Maneesh Agrawala, Stanford University) compresses historical video context frames into a **constant-length transformer token budget** (`1 + 2 + 16` multi-scale clean context latents + `9` active window latents). As a result, generating a **60-second (1,800-frame @ 30 FPS) video** requires the **exact same GPU VRAM** as generating a 2-second clip!

### ✨ Key Features of This Colab Notebook
- **Both Official FramePack Architectures in One Notebook**:
  - **`FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)`** (`lllyasviel/FramePack_F1_I2V_HY_20250503`) — pure causal forward temporal generation (`t = 0 → T`), eliminating backward-pass temporal jumps while delivering higher motion dynamism.
  - **`FramePack Original (Bidirectional Anti-Drifting)`** (`lllyasviel/FramePackI2V_HY`) — early-established endpoint conditioning that anchors both the start frame and late frames to prevent identity drift across ultra-long 30s–60s clips.
- **Smart Selective Drive Cache (Saves ~26 GB of Redundant Downloads)**:
  - Downloads only `text_encoder`, `text_encoder_2`, `tokenizer`, `tokenizer_2`, and `vae` from `hunyuanvideo-community/HunyuanVideo` (skipping the unused 26 GB base HunyuanVideo transformer), plus `lllyasviel/flux_redux_bfl` SigLIP vision encoder and the packed 13B FramePack transformer.
- **`DynamicSwapInstaller` + `gpu_memory_preservation` + TeaCache**:
  - Streams 13B transformer layers between pinned host RAM and GPU VRAM 3× faster than standard sequential offload, enabling **30 FPS** video generation on **T4 (16 GB)**, **L4 (24 GB)**, and **A100 (40/80 GB)**.

| GPU Tier | VRAM | Recommended `gpu_memory_preservation` | TeaCache | Supported Duration (30 FPS) |
| :--- | :--- | :--- | :--- | :--- |
| **Colab T4** | 16 GB | `8.0` – `10.0 GB` (DynamicSwap) | Enabled (`~1.8×` faster) | `1s` – `30s+` (`480p` / `640p` bucket) |
| **Colab L4** | 24 GB | `6.0 GB` (DynamicSwap) | Optional (`True` for speed, `False` for max detail) | `1s` – `60s` (`640p` bucket, up to 1,800 frames) |
| **Colab A100** | 40 / 80 GB | `6.0 GB` (or High-VRAM full resident on 80 GB) | Optional | `1s` – `60s` (`640p` / `720p` bucket, up to 1,800 frames) |

In [ ]:
# @title STEP 1 — Clone FramePack & Install Dependencies
# @markdown Clones `lllyasviel/FramePack` into `/content/FramePack`, installs `diffusers`, `transformers`, `accelerate`,
# @markdown `torchsde`, `einops`, `av`, `tqdm==4.66.5`, and `gradio==5.49.1`, and configures GPU memory defaults.

import os, sys, subprocess, time, shutil, torch

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Go to Runtime -> Change runtime type -> select T4, L4, or A100 GPU.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB VRAM | PyTorch {torch.__version__} (CUDA {torch.version.cuda})')

FRAMEPACK_REPO_DIR = '/content/FramePack'
if not os.path.exists(os.path.join(FRAMEPACK_REPO_DIR, 'diffusers_helper', 'hunyuan.py')):
    print(f'[Git] Cloning https://github.com/lllyasviel/FramePack.git -> {FRAMEPACK_REPO_DIR} ...')
    if os.path.exists(FRAMEPACK_REPO_DIR):
        shutil.rmtree(FRAMEPACK_REPO_DIR)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/lllyasviel/FramePack.git',
        FRAMEPACK_REPO_DIR,
    ], check=True)
else:
    print(f'[Git] FramePack repository already present at {FRAMEPACK_REPO_DIR}')

if FRAMEPACK_REPO_DIR not in sys.path:
    sys.path.insert(0, FRAMEPACK_REPO_DIR)

t0 = time.time()
print('[pip] Installing FramePack dependencies + gradio==5.49.1 + tqdm==4.66.5 ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.33.1',
    'transformers>=4.46.2',
    'accelerate>=1.6.0',
    'huggingface-hub>=0.27.0',
    'torchsde>=0.2.6',
    'einops',
    'safetensors',
    'opencv-contrib-python-headless',
    'av',
    'sentencepiece',
    'scipy',
    'tqdm==4.66.5',
    'gradio==5.49.1',
], check=True)
print(f'[Install] Completed in {time.time() - t0:.1f}s.')

if vram_gb < 20.0:
    DEFAULT_GPU_PRESERVE_GB = 8.0
    DEFAULT_RES_BUCKET = 512
elif vram_gb < 35.0:
    DEFAULT_GPU_PRESERVE_GB = 6.0
    DEFAULT_RES_BUCKET = 640
else:
    DEFAULT_GPU_PRESERVE_GB = 6.0
    DEFAULT_RES_BUCKET = 640

print(f'[Auto-Tune] Default gpu_memory_preservation = {DEFAULT_GPU_PRESERVE_GB} GB | resolution bucket = {DEFAULT_RES_BUCKET}p')

In [ ]:
# @title STEP 2 — Mount Google Drive & Pre-Cache FramePack Weights (Selective Download)
# @markdown Mounts Google Drive and caches shared HunyuanVideo text/VAE encoders, Flux Redux SigLIP image encoder,
# @markdown and the packed FramePack 13B transformer under `/content/drive/MyDrive/AEI_3D_Cache/FramePack`.

import os, sys, time, pathlib, shutil
from huggingface_hub import snapshot_download
from tqdm.notebook import tqdm

FRAMEPACK_MODEL_REGISTRY = {
    'FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)': (
        'lllyasviel/FramePack_F1_I2V_HY_20250503',
        'f1',
        'Forward-only causal next-frame prediction (drift-free, high motion dynamism)',
    ),
    'FramePack Original (Bidirectional Anti-Drifting)': (
        'lllyasviel/FramePackI2V_HY',
        'original',
        'Bidirectional inverted anti-drifting sampling (anchors endpoints first)',
    ),
}

HUNYUAN_BASE_REPO = 'hunyuanvideo-community/HunyuanVideo'
FLUX_REDUX_REPO = 'lllyasviel/flux_redux_bfl'

DEFAULT_FRAMEPACK_VARIANT = 'FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)'  # @param ['FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)', 'FramePack Original (Bidirectional Anti-Drifting)']
PRECACHE_ON_STEP2 = True  # @param {type:"boolean"}

try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/FramePack')
    HAS_DRIVE = True
except Exception as exc:
    print(f'[Cache] Google Drive not available ({exc}); using /content/hf_cache/FramePack.')
    CACHE_ROOT = pathlib.Path('/content/hf_cache/FramePack')
    HAS_DRIVE = False

HF_HOME_DIR = CACHE_ROOT / 'hf_home'
HF_HUB_DIR = HF_HOME_DIR / 'hub'
HF_HUB_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HUB_DIR)
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

OUT_DIR = pathlib.Path('/content/framepack_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/FramePack')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_OUT = OUT_DIR

def precache_framepack_weights(variant_key: str):
    if variant_key not in FRAMEPACK_MODEL_REGISTRY:
        raise ValueError(f'Unknown FramePack variant: {variant_key}')
    dit_repo, _, blurb = FRAMEPACK_MODEL_REGISTRY[variant_key]
    t0 = time.time()
    print('[Cache 1/3] Caching shared HunyuanVideo encoders (text_encoder, text_encoder_2, tokenizer, vae — skipping 26GB base transformer) ...')
    snapshot_download(
        repo_id=HUNYUAN_BASE_REPO,
        allow_patterns=['text_encoder/*', 'text_encoder_2/*', 'tokenizer/*', 'tokenizer_2/*', 'vae/*', 'model_index.json', 'config.json'],
        tqdm_class=tqdm,
    )
    print('[Cache 2/3] Caching SigLIP Vision Encoder (lllyasviel/flux_redux_bfl) ...')
    snapshot_download(
        repo_id=FLUX_REDUX_REPO,
        allow_patterns=['feature_extractor/*', 'image_encoder/*'],
        tqdm_class=tqdm,
    )
    print(f'[Cache 3/3] Caching {variant_key} ({dit_repo}) — {blurb} ...')
    snapshot_download(
        repo_id=dit_repo,
        tqdm_class=tqdm,
    )
    print(f'[Cache] All FramePack weights ready in {time.time() - t0:.1f}s.')

print(f"[Setup] HF_HOME    : {os.environ['HF_HOME']}")
print(f'[Setup] Output dir : {DRIVE_OUT}')
if PRECACHE_ON_STEP2:
    precache_framepack_weights(DEFAULT_FRAMEPACK_VARIANT)
else:
    print('[Setup] Pre-download skipped; weights will download on demand in Step 3/4.')

In [ ]:
# @title STEP 3 — FramePack & FramePack-F1 Core Engine (`DynamicSwap` + TeaCache + Constant VRAM)
# @markdown Implements `FramePackEngine` using upstream `diffusers_helper` modules with support for
# @markdown both **`FramePack-F1`** (forward next-frame prediction) and **`FramePack Original`** (bidirectional anti-drifting).

import os, sys, time, gc, math, uuid, random, shutil, pathlib, traceback
from typing import Callable, Optional, Tuple
import torch, einops, numpy as np
from PIL import Image
from IPython.display import display, FileLink

from diffusers import AutoencoderKLHunyuanVideo
from transformers import LlamaModel, CLIPTextModel, LlamaTokenizerFast, CLIPTokenizer, SiglipImageProcessor, SiglipVisionModel
from diffusers_helper.hunyuan import encode_prompt_conds, vae_decode, vae_encode, vae_decode_fake
from diffusers_helper.utils import save_bcthw_as_mp4, crop_or_pad_yield_mask, soft_append_bcthw, resize_and_center_crop, generate_timestamp
from diffusers_helper.models.hunyuan_video_packed import HunyuanVideoTransformer3DModelPacked
from diffusers_helper.pipelines.k_diffusion_hunyuan import sample_hunyuan
from diffusers_helper.memory import (
    cpu, gpu, get_cuda_free_memory_gb,
    move_model_to_device_with_memory_preservation,
    offload_model_from_device_for_memory_preservation,
    fake_diffusers_current_device,
    DynamicSwapInstaller,
    unload_complete_models,
    load_model_as_complete,
)
from diffusers_helper.clip_vision import hf_clip_vision_encode
from diffusers_helper.bucket_tools import find_nearest_bucket

class FramePackEngine:
    """Unified engine for FramePack-F1 and Original FramePack with DynamicSwap constant-VRAM streaming."""
    def __init__(self):
        self.encoders_loaded = False
        self.active_variant = None
        self.high_vram = (get_cuda_free_memory_gb(gpu) > 60.0)
        self.text_encoder = None
        self.text_encoder_2 = None
        self.tokenizer = None
        self.tokenizer_2 = None
        self.vae = None
        self.feature_extractor = None
        self.image_encoder = None
        self.transformer = None

    def _load_shared_encoders(self):
        if self.encoders_loaded:
            return
        print('[FramePack] Loading shared HunyuanVideo text encoders, Causal 3D VAE, and SigLIP Vision Encoder ...')
        t0 = time.time()
        self.text_encoder = LlamaModel.from_pretrained(HUNYUAN_BASE_REPO, subfolder='text_encoder', torch_dtype=torch.float16).cpu()
        self.text_encoder_2 = CLIPTextModel.from_pretrained(HUNYUAN_BASE_REPO, subfolder='text_encoder_2', torch_dtype=torch.float16).cpu()
        self.tokenizer = LlamaTokenizerFast.from_pretrained(HUNYUAN_BASE_REPO, subfolder='tokenizer')
        self.tokenizer_2 = CLIPTokenizer.from_pretrained(HUNYUAN_BASE_REPO, subfolder='tokenizer_2')
        self.vae = AutoencoderKLHunyuanVideo.from_pretrained(HUNYUAN_BASE_REPO, subfolder='vae', torch_dtype=torch.float16).cpu()
        self.feature_extractor = SiglipImageProcessor.from_pretrained(FLUX_REDUX_REPO, subfolder='feature_extractor')
        self.image_encoder = SiglipVisionModel.from_pretrained(FLUX_REDUX_REPO, subfolder='image_encoder', torch_dtype=torch.float16).cpu()

        for m in (self.vae, self.text_encoder, self.text_encoder_2, self.image_encoder):
            m.eval()
            m.requires_grad_(False)

        if not self.high_vram:
            self.vae.enable_slicing()
            self.vae.enable_tiling()
            DynamicSwapInstaller.install_model(self.text_encoder, device=gpu)
        else:
            self.text_encoder.to(gpu)
            self.text_encoder_2.to(gpu)
            self.image_encoder.to(gpu)
            self.vae.to(gpu)

        self.encoders_loaded = True
        print(f'[FramePack] Shared encoders ready in {time.time() - t0:.1f}s (high_vram={self.high_vram}).')

    def load(self, variant_key: str = DEFAULT_FRAMEPACK_VARIANT, force_reload: bool = False):
        if variant_key not in FRAMEPACK_MODEL_REGISTRY:
            raise ValueError(f'Unknown variant: {variant_key}')
        self._load_shared_encoders()
        if self.transformer is not None and self.active_variant == variant_key and not force_reload:
            return
        if self.transformer is not None:
            del self.transformer
            self.transformer = None
            gc.collect()
            torch.cuda.empty_cache()

        dit_repo, arch_mode, _ = FRAMEPACK_MODEL_REGISTRY[variant_key]
        print(f'[FramePack] Loading packed 13B transformer from {dit_repo} ({arch_mode}) ...')
        t0 = time.time()
        self.transformer = HunyuanVideoTransformer3DModelPacked.from_pretrained(
            dit_repo, torch_dtype=torch.bfloat16
        ).cpu()
        self.transformer.eval()
        self.transformer.high_quality_fp32_output_for_inference = True
        self.transformer.to(dtype=torch.bfloat16)
        self.transformer.requires_grad_(False)
        if not self.high_vram:
            DynamicSwapInstaller.install_model(self.transformer, device=gpu)
        else:
            self.transformer.to(gpu)
        self.active_variant = variant_key
        print(f'[FramePack] Transformer ready in {time.time() - t0:.1f}s.')

    def unload(self):
        for attr in ('transformer', 'text_encoder', 'text_encoder_2', 'image_encoder', 'vae'):
            obj = getattr(self, attr, None)
            if obj is not None:
                delattr(self, attr)
                setattr(self, attr, None)
        self.encoders_loaded = False
        self.active_variant = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print('[FramePack] Unloaded all models from RAM/VRAM.')

    @torch.no_grad()
    def generate(
        self,
        input_image,
        prompt: str,
        negative_prompt: str = '',
        variant_key: str = DEFAULT_FRAMEPACK_VARIANT,
        total_second_length: float = 5.0,
        latent_window_size: int = 9,
        steps: int = 25,
        cfg: float = 1.0,
        gs: float = 10.0,
        rs: float = 0.0,
        gpu_memory_preservation: float = 6.0,
        use_teacache: bool = True,
        mp4_crf: int = 16,
        resolution_bucket: int = 640,
        seed: int = 42,
        on_section_callback: Optional[Callable[[str, Optional[np.ndarray], str], None]] = None,
    ) -> str:
        self.load(variant_key)
        _, arch_mode, _ = FRAMEPACK_MODEL_REGISTRY[variant_key]

        if isinstance(input_image, (str, pathlib.Path)):
            input_image_np = np.array(Image.open(str(input_image)).convert('RGB'))
        elif isinstance(input_image, Image.Image):
            input_image_np = np.array(input_image.convert('RGB'))
        elif isinstance(input_image, np.ndarray):
            input_image_np = input_image
        else:
            raise ValueError('Please provide a valid input image (PIL, numpy array, or file path).')

        total_latent_sections = int(max(round((float(total_second_length) * 30.0) / (int(latent_window_size) * 4)), 1))
        actual_seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31 - 1)
        job_id = generate_timestamp() + f'_{uuid.uuid4().hex[:4]}'

        if not self.high_vram:
            unload_complete_models(self.text_encoder, self.text_encoder_2, self.image_encoder, self.vae, self.transformer)
            fake_diffusers_current_device(self.text_encoder, gpu)
            load_model_as_complete(self.text_encoder_2, target_device=gpu)

        llama_vec, clip_l_pooler = encode_prompt_conds(
            prompt, self.text_encoder, self.text_encoder_2, self.tokenizer, self.tokenizer_2
        )
        if float(cfg) == 1.0:
            llama_vec_n, clip_l_pooler_n = torch.zeros_like(llama_vec), torch.zeros_like(clip_l_pooler)
        else:
            llama_vec_n, clip_l_pooler_n = encode_prompt_conds(
                negative_prompt, self.text_encoder, self.text_encoder_2, self.tokenizer, self.tokenizer_2
            )

        llama_vec, llama_attention_mask = crop_or_pad_yield_mask(llama_vec, length=512)
        llama_vec_n, llama_attention_mask_n = crop_or_pad_yield_mask(llama_vec_n, length=512)

        H, W, _ = input_image_np.shape
        height, width = find_nearest_bucket(H, W, resolution=int(resolution_bucket))
        input_image_cropped = resize_and_center_crop(input_image_np, target_width=width, target_height=height)
        input_image_pt = torch.from_numpy(input_image_cropped).float() / 127.5 - 1.0
        input_image_pt = input_image_pt.permute(2, 0, 1)[None, :, None]

        if not self.high_vram:
            load_model_as_complete(self.vae, target_device=gpu)
        start_latent = vae_encode(input_image_pt, self.vae)

        if not self.high_vram:
            load_model_as_complete(self.image_encoder, target_device=gpu)
        image_encoder_output = hf_clip_vision_encode(input_image_cropped, self.feature_extractor, self.image_encoder)
        image_encoder_last_hidden_state = image_encoder_output.last_hidden_state

        llama_vec = llama_vec.to(self.transformer.dtype)
        llama_vec_n = llama_vec_n.to(self.transformer.dtype)
        clip_l_pooler = clip_l_pooler.to(self.transformer.dtype)
        clip_l_pooler_n = clip_l_pooler_n.to(self.transformer.dtype)
        image_encoder_last_hidden_state = image_encoder_last_hidden_state.to(self.transformer.dtype)

        rnd = torch.Generator('cpu').manual_seed(actual_seed)
        num_frames = int(latent_window_size) * 4 - 3
        output_filename = None

        if arch_mode == 'f1':
            history_latents = torch.zeros(size=(1, 16, 16 + 2 + 1, height // 8, width // 8), dtype=torch.float32).cpu()
            history_pixels = None
            history_latents = torch.cat([history_latents, start_latent.to(history_latents)], dim=2)
            total_generated_latent_frames = 1

            for section_index in range(total_latent_sections):
                if not self.high_vram:
                    unload_complete_models()
                    move_model_to_device_with_memory_preservation(
                        self.transformer, target_device=gpu, preserved_memory_gb=float(gpu_memory_preservation)
                    )
                self.transformer.initialize_teacache(enable_teacache=bool(use_teacache), num_steps=int(steps))

                indices = torch.arange(0, sum([1, 16, 2, 1, int(latent_window_size)])).unsqueeze(0)
                clean_latent_indices_start, clean_latent_4x_indices, clean_latent_2x_indices, clean_latent_1x_indices, latent_indices = indices.split(
                    [1, 16, 2, 1, int(latent_window_size)], dim=1
                )
                clean_latent_indices = torch.cat([clean_latent_indices_start, clean_latent_1x_indices], dim=1)
                clean_latents_4x, clean_latents_2x, clean_latents_1x = history_latents[:, :, -sum([16, 2, 1]):, :, :].split([16, 2, 1], dim=2)
                clean_latents = torch.cat([start_latent.to(history_latents), clean_latents_1x], dim=2)

                generated_latents = sample_hunyuan(
                    transformer=self.transformer,
                    sampler='unipc',
                    width=width,
                    height=height,
                    frames=num_frames,
                    real_guidance_scale=float(cfg),
                    distilled_guidance_scale=float(gs),
                    guidance_rescale=float(rs),
                    num_inference_steps=int(steps),
                    generator=rnd,
                    prompt_embeds=llama_vec,
                    prompt_embeds_mask=llama_attention_mask,
                    prompt_poolers=clip_l_pooler,
                    negative_prompt_embeds=llama_vec_n,
                    negative_prompt_embeds_mask=llama_attention_mask_n,
                    negative_prompt_poolers=clip_l_pooler_n,
                    device=gpu,
                    dtype=torch.bfloat16,
                    image_embeddings=image_encoder_last_hidden_state,
                    latent_indices=latent_indices,
                    clean_latents=clean_latents,
                    clean_latent_indices=clean_latent_indices,
                    clean_latents_2x=clean_latents_2x,
                    clean_latent_2x_indices=clean_latent_2x_indices,
                    clean_latents_4x=clean_latents_4x,
                    clean_latent_4x_indices=clean_latent_4x_indices,
                )

                total_generated_latent_frames += int(generated_latents.shape[2])
                history_latents = torch.cat([history_latents, generated_latents.to(history_latents)], dim=2)

                if not self.high_vram:
                    offload_model_from_device_for_memory_preservation(self.transformer, target_device=gpu, preserved_memory_gb=8)
                    load_model_as_complete(self.vae, target_device=gpu)

                real_history_latents = history_latents[:, :, -total_generated_latent_frames:, :, :]
                if history_pixels is None:
                    history_pixels = vae_decode(real_history_latents, self.vae).cpu()
                else:
                    section_latent_frames = int(latent_window_size) * 2
                    overlapped_frames = int(latent_window_size) * 4 - 3
                    current_pixels = vae_decode(real_history_latents[:, :, -section_latent_frames:], self.vae).cpu()
                    history_pixels = soft_append_bcthw(history_pixels, current_pixels, overlapped_frames)

                if not self.high_vram:
                    unload_complete_models()

                output_filename = str(OUT_DIR / f'framepack_f1_{job_id}_{total_generated_latent_frames}.mp4')
                save_bcthw_as_mp4(history_pixels, output_filename, fps=30, crf=int(mp4_crf))
                if on_section_callback:
                    on_section_callback(output_filename, None, f'Section {section_index + 1}/{total_latent_sections} decoded ({history_pixels.shape[2]} frames @ 30fps)')
        else:
            history_latents = torch.zeros(size=(1, 16, 1 + 2 + 16, height // 8, width // 8), dtype=torch.float32).cpu()
            history_pixels = None
            total_generated_latent_frames = 0
            if total_latent_sections > 4:
                latent_paddings = [3] + [2] * (total_latent_sections - 3) + [1, 0]
            else:
                latent_paddings = list(reversed(range(total_latent_sections)))

            for sec_idx, latent_padding in enumerate(latent_paddings):
                is_last_section = (latent_padding == 0)
                latent_padding_size = int(latent_padding) * int(latent_window_size)

                if not self.high_vram:
                    unload_complete_models()
                    move_model_to_device_with_memory_preservation(
                        self.transformer, target_device=gpu, preserved_memory_gb=float(gpu_memory_preservation)
                    )
                self.transformer.initialize_teacache(enable_teacache=bool(use_teacache), num_steps=int(steps))

                indices = torch.arange(0, sum([1, latent_padding_size, int(latent_window_size), 1, 2, 16])).unsqueeze(0)
                clean_latent_indices_pre, blank_indices, latent_indices, clean_latent_indices_post, clean_latent_2x_indices, clean_latent_4x_indices = indices.split(
                    [1, latent_padding_size, int(latent_window_size), 1, 2, 16], dim=1
                )
                clean_latent_indices = torch.cat([clean_latent_indices_pre, clean_latent_indices_post], dim=1)
                clean_latents_pre = start_latent.to(history_latents)
                clean_latents_post, clean_latents_2x, clean_latents_4x = history_latents[:, :, :1 + 2 + 16, :, :].split([1, 2, 16], dim=2)
                clean_latents = torch.cat([clean_latents_pre, clean_latents_post], dim=2)

                generated_latents = sample_hunyuan(
                    transformer=self.transformer,
                    sampler='unipc',
                    width=width,
                    height=height,
                    frames=num_frames,
                    real_guidance_scale=float(cfg),
                    distilled_guidance_scale=float(gs),
                    guidance_rescale=float(rs),
                    num_inference_steps=int(steps),
                    generator=rnd,
                    prompt_embeds=llama_vec,
                    prompt_embeds_mask=llama_attention_mask,
                    prompt_poolers=clip_l_pooler,
                    negative_prompt_embeds=llama_vec_n,
                    negative_prompt_embeds_mask=llama_attention_mask_n,
                    negative_prompt_poolers=clip_l_pooler_n,
                    device=gpu,
                    dtype=torch.bfloat16,
                    image_embeddings=image_encoder_last_hidden_state,
                    latent_indices=latent_indices,
                    clean_latents=clean_latents,
                    clean_latent_indices=clean_latent_indices,
                    clean_latents_2x=clean_latents_2x,
                    clean_latent_2x_indices=clean_latent_2x_indices,
                    clean_latents_4x=clean_latents_4x,
                    clean_latent_4x_indices=clean_latent_4x_indices,
                )

                if is_last_section:
                    generated_latents = torch.cat([start_latent.to(generated_latents), generated_latents], dim=2)

                total_generated_latent_frames += int(generated_latents.shape[2])
                history_latents = torch.cat([generated_latents.to(history_latents), history_latents], dim=2)

                if not self.high_vram:
                    offload_model_from_device_for_memory_preservation(self.transformer, target_device=gpu, preserved_memory_gb=8)
                    load_model_as_complete(self.vae, target_device=gpu)

                real_history_latents = history_latents[:, :, :total_generated_latent_frames, :, :]
                if history_pixels is None:
                    history_pixels = vae_decode(real_history_latents, self.vae).cpu()
                else:
                    section_latent_frames = (int(latent_window_size) * 2 + 1) if is_last_section else (int(latent_window_size) * 2)
                    overlapped_frames = int(latent_window_size) * 4 - 3
                    current_pixels = vae_decode(real_history_latents[:, :, :section_latent_frames], self.vae).cpu()
                    history_pixels = soft_append_bcthw(current_pixels, history_pixels, overlapped_frames)

                if not self.high_vram:
                    unload_complete_models()

                output_filename = str(OUT_DIR / f'framepack_orig_{job_id}_{total_generated_latent_frames}.mp4')
                save_bcthw_as_mp4(history_pixels, output_filename, fps=30, crf=int(mp4_crf))
                if on_section_callback:
                    on_section_callback(output_filename, None, f'Section {sec_idx + 1}/{len(latent_paddings)} decoded ({history_pixels.shape[2]} frames @ 30fps)')
                if is_last_section:
                    break

        final_drive_mp4 = DRIVE_OUT / pathlib.Path(output_filename).name
        shutil.copy2(output_filename, str(final_drive_mp4))
        return str(final_drive_mp4)

ENGINE = FramePackEngine()
print('[Core] FramePackEngine ready. Supported variants:', list(FRAMEPACK_MODEL_REGISTRY.keys()))

In [ ]:
# @title STEP 4 — Launch Gradio UI (`FramePack-F1` & `Original FramePack` 60s Studio)
# @markdown Launches the Gradio interface with `FramePack-F1` vs `FramePack Original` selector,
# @markdown `total_second_length` (`1s` to `60s` @ 30 FPS), `TeaCache` toggle, and `gpu_memory_preservation` slider.

import os, sys, time, shutil, pathlib, traceback
import torch, numpy as np, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

def ui_generate_framepack(
    variant_key, input_image, prompt, neg_prompt,
    total_seconds, res_bucket, steps, gs, cfg, rs,
    use_teacache, gpu_preserve, latent_window_size, mp4_crf, seed,
    progress=gr.Progress(),
):
    if input_image is None:
        return None, '❌ Please upload a starting keyframe / portrait image.'
    if not prompt or not prompt.strip():
        return None, '❌ Please enter a motion prompt.'
    try:
        t0 = time.time()
        progress(0.05, desc=f'Loading {variant_key} & shared HunyuanVideo encoders ...')
        out_mp4 = ENGINE.generate(
            input_image=input_image,
            prompt=prompt.strip(),
            negative_prompt=neg_prompt or '',
            variant_key=variant_key,
            total_second_length=float(total_seconds),
            latent_window_size=int(latent_window_size),
            steps=int(steps),
            cfg=float(cfg),
            gs=float(gs),
            rs=float(rs),
            gpu_memory_preservation=float(gpu_preserve),
            use_teacache=bool(use_teacache),
            mp4_crf=int(mp4_crf),
            resolution_bucket=int(res_bucket),
            seed=int(seed),
        )
        progress(1.0, desc='Done!')
        return out_mp4, f'✅ **Generated {float(total_seconds):.1f}s @ 30 FPS** in **{time.time() - t0:.1f}s** ({variant_key}) → `{out_mp4}`'
    except Exception as exc:
        traceback.print_exc()
        return None, f'❌ Error: {exc}'

def vram_status():
    if not torch.cuda.is_available():
        return 'No CUDA GPU.'
    a = torch.cuda.memory_allocated() / (1024 ** 3)
    r = torch.cuda.memory_reserved() / (1024 ** 3)
    t = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    return f"GPU: {torch.cuda.get_device_name(0)}\nAllocated: {a:.2f} GB | Reserved: {r:.2f} GB | Total: {t:.2f} GB\nActive Variant: {ENGINE.active_variant or 'None'}"

def free_vram():
    ENGINE.unload()
    return vram_status()

with gr.Blocks(title='FramePack & FramePack-F1 Studio', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# 🎞️ FramePack & FramePack-F1 — Constant-VRAM Next-Frame-Prediction Video Studio\n'
        'Generate **1s to 60s (up to 1,800 frames @ 30 FPS)** Image-to-Video animations with constant VRAM usage.'
    )

    with gr.Tabs():
        with gr.Tab('1. 🎬 Generate Video (FramePack-F1 / Original)'):
            with gr.Row():
                with gr.Column(scale=3):
                    variant_dd = gr.Dropdown(
                        choices=list(FRAMEPACK_MODEL_REGISTRY.keys()),
                        value=DEFAULT_FRAMEPACK_VARIANT,
                        label='FramePack Model Architecture',
                        info='FramePack-F1 uses forward-only next-frame prediction (recommended). FramePack Original uses bidirectional anti-drifting.',
                    )
                    img_in = gr.Image(
                        type='numpy', label='Start Frame / Character Portrait', height=280,
                    )
                    prompt_in = gr.Textbox(
                        label='Motion & Action Prompt',
                        value='The character dances gracefully with fluid natural arm and body movements, warm studio lighting, smooth camera tracking.',
                        lines=3,
                        info='Describe the subject motion, facial expression, and camera movement.',
                    )
                    with gr.Row():
                        duration_sl = gr.Slider(
                            1.0, 60.0, value=5.0, step=1.0, label='Total Video Duration (seconds @ 30 FPS)',
                            info='5s = 150 frames (~4 sections); 10s = 300 frames; 60s = 1,800 frames with constant VRAM.',
                        )
                        res_bucket_dd = gr.Dropdown(
                            choices=[480, 512, 640, 720],
                            value=DEFAULT_RES_BUCKET if 'DEFAULT_RES_BUCKET' in globals() else 640,
                            label='Resolution Bucket (Aspect-Ratio Preserving)',
                            info='640 is upstream standard; use 480 or 512 on T4 16GB for faster section sampling.',
                        )
                    with gr.Row():
                        teacache_cb = gr.Checkbox(
                            value=True, label='Enable TeaCache (~1.8× Faster Sampling)',
                            info='Caches intermediate DiT residuals across steps; disable slightly for maximum fine hand/finger detail.',
                        )
                        gpu_preserve_sl = gr.Slider(
                            4.0, 24.0, value=DEFAULT_GPU_PRESERVE_GB if 'DEFAULT_GPU_PRESERVE_GB' in globals() else 6.0, step=0.5,
                            label='GPU Memory Preservation (GB)',
                            info='Higher value leaves more free GPU VRAM by swapping more transformer blocks to CPU RAM. Use 8–10 on T4, 6 on L4/A100.',
                        )
                    with gr.Accordion('⚙️ Advanced Sampling Parameters', open=False):
                        with gr.Row():
                            steps_sl = gr.Slider(10, 50, value=25, step=1, label='Steps per Section', info='25 steps is the upstream default.')
                            gs_sl = gr.Slider(1.0, 32.0, value=10.0, step=0.5, label='Distilled Guidance Scale (gs)', info='10.0 is the upstream default for HunyuanVideo packed DiT.')
                            seed_sl = gr.Slider(-1, 2147483647, value=42, step=1, label='Seed (-1 for random)', info='Fixed integer seed for reproducible motion.')
                        with gr.Row():
                            cfg_sl = gr.Slider(1.0, 5.0, value=1.0, step=0.5, label='Real Guidance Scale (CFG)', info='Keep at 1.0 (distilled guidance handles prompt adherence without doubling forward passes).')
                            rs_sl = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label='Guidance Rescale', info='Default 0.0.')
                            window_sl = gr.Slider(5, 9, value=9, step=4, label='Latent Window Size', info='9 latent frames = 33 pixel frames per section (upstream default; do not change unless testing).')
                            crf_sl = gr.Slider(0, 35, value=16, step=1, label='MP4 CRF Quality', info='Lower = higher quality MP4 (16 is near-lossless).')
                        neg_in = gr.Textbox(label='Negative Prompt (used only if Real CFG > 1.0)', value='', lines=1, info='Optional negative prompt when Real CFG > 1.0.')
                    gen_btn = gr.Button('🎞️ Generate Video', variant='primary')
                with gr.Column(scale=2):
                    vid_out = gr.Video(label='Generated 30 FPS MP4 Video', height=420)
                    status_md = gr.Markdown('Ready.')
            gen_btn.click(
                ui_generate_framepack,
                inputs=[
                    variant_dd, img_in, prompt_in, neg_in,
                    duration_sl, res_bucket_dd, steps_sl, gs_sl, cfg_sl, rs_sl,
                    teacache_cb, gpu_preserve_sl, window_sl, crf_sl, seed_sl,
                ],
                outputs=[vid_out, status_md],
            )

        with gr.Tab('2. 🧠 VRAM & Engine Status'):
            vram_box = gr.Textbox(label='GPU Memory & Model Status', value=vram_status, lines=4, interactive=False, info='Displays current GPU VRAM usage and loaded FramePack architecture.')
            with gr.Row():
                gr.Button('🔄 Refresh Status').click(vram_status, outputs=[vram_box])
                gr.Button('🧹 Unload Models & Free VRAM').click(free_vram, outputs=[vram_box])

clear_output()
print('[UI] Launching FramePack & FramePack-F1 Studio ...')
demo.queue(max_size=4, concurrency_limit=1)
demo.load(lambda: '✅ **FramePack Studio ready** — upload a start frame, pick `FramePack-F1` or `Original`, and click Generate.', outputs=[status_md])
demo.launch(share=True, show_error=True, height=880, prevent_thread_lock=False)

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings Hugging Face API every 60 seconds in a daemon thread to prevent idle disconnects during 30s–60s video renders.

import time, threading, datetime, requests, IPython
_stop_keepalive = threading.Event()

def _pinger():
    while not _stop_keepalive.is_set():
        try:
            requests.get('https://huggingface.co/api/models/lllyasviel/FramePack_F1_I2V_HY_20250503', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f"[Keep-Alive] {datetime.datetime.now().strftime('%H:%M:%S')} OK — stop with `_stop_keepalive.set()`.")
        except Exception as exc:
            print(f'[Keep-Alive] Warning: {exc}')
        _stop_keepalive.wait(60)

_ka_thread = threading.Thread(target=_pinger, daemon=True)
_ka_thread.start()
print('[Keep-Alive] Daemon started.')

In [ ]:
# @title STEP 6 — Quick Test (Headless 1-Section FramePack-F1 Smoke Test)
# @markdown Runs a single-section (`~1.1s` @ 30 FPS) generation to verify all encoders, `DynamicSwap`, and MP4 export.

import os, time, traceback
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, FileLink

QUICK_VARIANT = 'FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)'  # @param ['FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)', 'FramePack Original (Bidirectional Anti-Drifting)']
QUICK_PROMPT = 'A golden lantern swaying gently in the evening breeze with warm glowing sparks.'  # @param {type:"string"}
QUICK_STEPS = 15  # @param {type:"integer"}

try:
    canvas = Image.new('RGB', (512, 512), (24, 32, 64))
    draw = ImageDraw.Draw(canvas)
    draw.ellipse((176, 156, 336, 356), fill=(255, 190, 70), outline=(255, 240, 180), width=4)
    t0 = time.time()
    out_mp4 = ENGINE.generate(
        input_image=canvas,
        prompt=QUICK_PROMPT,
        variant_key=QUICK_VARIANT,
        total_second_length=1.0,
        steps=QUICK_STEPS,
        gpu_memory_preservation=DEFAULT_GPU_PRESERVE_GB,
        use_teacache=True,
        resolution_bucket=480,
        seed=42,
    )
    print(f'[QuickTest] Completed in {time.time() - t0:.1f}s -> {out_mp4}')
    display(FileLink(str(out_mp4)))
except Exception as exc:
    traceback.print_exc()
    raise

In [ ]:
# @title STEP 7 — Batch Image-to-Video Generation
# @markdown Iterates over a folder of input images (`.png` / `.jpg`) and generates a 30 FPS `.mp4` for each
# @markdown with per-item `try`/`except` error isolation.

import os, time, pathlib, traceback
from tqdm import tqdm
from PIL import Image, ImageDraw
from IPython.display import display, FileLink

BATCH_VARIANT = 'FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)'  # @param ['FramePack-F1 (Forward-Only Next-Frame Prediction — Recommended)', 'FramePack Original (Bidirectional Anti-Drifting)']
BATCH_IMAGE_DIR = '/content/framepack_batch_inputs'  # @param {type:"string"}
BATCH_PROMPT = 'Natural cinematic motion, subtle camera movement, high quality 30fps video.'  # @param {type:"string"}
BATCH_SECONDS = 3.0  # @param {type:"number"}
BATCH_STEPS = 20  # @param {type:"integer"}

in_dir = pathlib.Path(BATCH_IMAGE_DIR)
in_dir.mkdir(parents=True, exist_ok=True)
images = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp')])
if not images:
    sample_img = in_dir / 'sample_card.png'
    im = Image.new('RGB', (512, 512), (35, 45, 75))
    ImageDraw.Draw(im).rectangle((120, 120, 392, 392), fill=(220, 160, 90))
    im.save(sample_img)
    images = [sample_img]
    print(f'[Batch] Created sample input image at {sample_img}')

ok_count, fail_count = 0, 0
for idx, img_path in enumerate(tqdm(images, desc='FramePack Batch'), 1):
    try:
        t0 = time.time()
        out_mp4 = ENGINE.generate(
            input_image=str(img_path),
            prompt=BATCH_PROMPT,
            variant_key=BATCH_VARIANT,
            total_second_length=float(BATCH_SECONDS),
            steps=int(BATCH_STEPS),
            gpu_memory_preservation=DEFAULT_GPU_PRESERVE_GB,
            use_teacache=True,
            resolution_bucket=DEFAULT_RES_BUCKET,
            seed=42 + idx,
        )
        print(f'  [{idx:02d}/{len(images)}] OK ({time.time() - t0:.1f}s) -> {out_mp4}')
        display(FileLink(str(out_mp4)))
        ok_count += 1
    except Exception as exc:
        fail_count += 1
        print(f'  [{idx:02d}/{len(images)}] FAILED on {img_path.name}: {exc}')
        traceback.print_exc()

print(f'[Batch] Finished: {ok_count} succeeded, {fail_count} failed. Output folder: {DRIVE_OUT}')